# Notebook 06 â€” StratLake Feature Validation, Archive, and Handoff

This notebook is a **standalone validation and handoff checkpoint** for the latest Fintech â†’ StratLake notebook workflow.

It keeps the same rules introduced in Notebook 04 and used in Notebook 05:

```text
FINTECH_SESSION_ID   -> upstream curated market-data session
STRATLAKE_SESSION_ID -> downstream StratLake feature/research session
MARKETLAKE_ROOT      -> explicit local Fintech curated-data handoff into StratLake
```

Notebook 06 is intentionally runnable from a fresh Colab runtime. It includes:

```text
package installs
Google Drive authorization
Alpaca credential setup
Fintech/StratLake session initialization
notebook config verification: universe.yml and paths.yml
optional daily-bars ingestion if local data is missing
optional feature generation if feature outputs are missing
manual command-surface previews that require follow-up registry verification
feature/data validation
Drive archive pack previews and optional archive execution
restore-readiness command previews
```

Boundary rule:

```text
Fintech curated data archive â‰  StratLake feature archive
Fintech session ID â‰  StratLake session ID
Drive backup/archive packs â‰  canonical active workspace data
MARKETLAKE_ROOT remains the explicit handoff from Fintech to StratLake
Command preview syntax is not treated as registry-validated in this import
```

## Install required packages

Run this cell in a fresh Colab runtime.

These installs intentionally mirror the earlier notebooks.

In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine

## Verify required CLI commands

This notebook uses Fintech session/backup commands and StratLake session/archive/feature commands.

If any command is missing, rerun the install cell above.

In [ ]:
import shutil

required_workflow_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-save-session",
    "fintech-restore-session",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-build-features",
    "stratlake-session-export",
    "stratlake-session-import",
]

optional_unverified_preview_commands = [
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

missing_required_commands = []
for command in required_workflow_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'NOT FOUND'}")
    if path is None:
        missing_required_commands.append(command)

print()
print("Optional/unverified preview commands:")
for command in optional_unverified_preview_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'NOT FOUND'}")

if missing_required_commands:
    raise RuntimeError(
        "Missing required workflow commands: " + ", ".join(missing_required_commands)
    )


## Authorize Google Drive access

Run this cell in Google Colab to authorize Google Drive access.

The packages do not mount Google Drive for you. Drive access is user-initiated here, and later commands treat Drive as a mounted filesystem path.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Define shared Fintech and StratLake paths

This notebook can either create fresh sessions or reconnect to known session IDs.

Default behavior is to create fresh local sessions so the notebook is independently runnable.

To reconnect to prior Notebook 05 sessions, set the override values below before running the session/path cells.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import textwrap

TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

FINTECH_ROOT = Path("/content/fintech-market-ingestion-demo")
STRATLAKE_ROOT = Path("/content/stratlake-trade-engine-demo")

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )
FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

# Explicit handoff from Fintech curated-data storage into StratLake.
MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"

# Keep names descriptive, but let the init commands own the final session ID.
FINTECH_SESSION_NAME = f"fintech_stratlake_validation_{TIMESTAMP_UTC}"
STRATLAKE_SESSION_NAME = f"stratlake_q1_feature_validation_{TIMESTAMP_UTC}"

# Optional: set these to prior Notebook 05 IDs to reconnect to existing Drive sessions.
# Leave as None for a fresh standalone run.
FINTECH_SESSION_ID_OVERRIDE = None
STRATLAKE_SESSION_ID_OVERRIDE = None

print("FINTECH_ROOT:", FINTECH_ROOT)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("FINTECH_SESSION_NAME:", FINTECH_SESSION_NAME)
print("STRATLAKE_SESSION_NAME:", STRATLAKE_SESSION_NAME)

## Initialize or reconnect the Fintech project session

This session provides the curated input data location for StratLake.

The preferred pattern is to discover `FINTECH_SESSION_ID` from the initialized session manifest. If you are reconnecting to a prior Notebook 05 run, use `FINTECH_SESSION_ID_OVERRIDE`.

In [ ]:
if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
    print("Using overridden FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
else:
    !fintech-init-project \
      --root {FINTECH_ROOT.as_posix()} \
      --notebooks \
      --with-session \
      --session-name {FINTECH_SESSION_NAME}

    fintech_manifest_paths = sorted(
        (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
        key=lambda path: path.stat().st_mtime,
    )

    if not fintech_manifest_paths:
        raise FileNotFoundError("No Fintech session_manifest.json files found.")

    FINTECH_SESSION_MANIFEST_PATH = fintech_manifest_paths[-1]
    FINTECH_SESSION_MANIFEST = json.loads(FINTECH_SESSION_MANIFEST_PATH.read_text(encoding="utf-8"))
    FINTECH_SESSION_ID = FINTECH_SESSION_MANIFEST["session_id"]

    print("FINTECH_SESSION_MANIFEST_PATH:", FINTECH_SESSION_MANIFEST_PATH)
    print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)

## Initialize or reconnect the StratLake project session

Use `--notebook-configs` so the notebook-oriented config bundle is available, including `configs/universe.yml` and `configs/paths.yml`.

The CLI does **not** accept arbitrary `--include-configs` or repeated `--config-file` arguments.

Use `--force-notebook-configs` only when intentionally refreshing existing generated notebook config files.

In [ ]:
if STRATLAKE_SESSION_ID_OVERRIDE:
    STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE
    print("Using overridden STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
else:
    !stratlake-init-session \
      --root {STRATLAKE_ROOT.as_posix()} \
      --project-name {STRATLAKE_SESSION_NAME} \
      --marketlake-root {MARKETLAKE_ROOT.as_posix()} \
      --drive-root {DRIVE_ROOT.as_posix()} \
      --enable-drive-persistence \
      --notebook-configs

    STRATLAKE_SESSION_FILE = STRATLAKE_ROOT / ".stratlake" / "session.json"

    if not STRATLAKE_SESSION_FILE.exists():
        raise FileNotFoundError(f"Missing StratLake session file: {STRATLAKE_SESSION_FILE}")

    STRATLAKE_SESSION_MANIFEST = json.loads(STRATLAKE_SESSION_FILE.read_text(encoding="utf-8"))
    STRATLAKE_SESSION_ID = (
        STRATLAKE_SESSION_MANIFEST.get("session_id")
        or STRATLAKE_SESSION_MANIFEST.get("project_name")
        or STRATLAKE_SESSION_NAME
    )

    print("STRATLAKE_SESSION_FILE:", STRATLAKE_SESSION_FILE)
    print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)

## Verify StratLake notebook config files

Confirm that `universe.yml` and `paths.yml` are present after initialization.

These config files should travel with the StratLake session config bundle and archive checkpoints.

In [ ]:
expected_notebook_configs = [
    STRATLAKE_ROOT / "configs" / "universe.yml",
    STRATLAKE_ROOT / "configs" / "paths.yml",
]

for config_path in expected_notebook_configs:
    print(f"{config_path}: {'FOUND' if config_path.exists() else 'MISSING'}")

missing_configs = [path for path in expected_notebook_configs if not path.exists()]
if missing_configs:
    raise FileNotFoundError(
        "Missing expected StratLake notebook config files: "
        + ", ".join(path.as_posix() for path in missing_configs)
    )

print("\nConfig preview:")
for config_path in expected_notebook_configs:
    print("\n---", config_path.name, "---")
    print(config_path.read_text(encoding="utf-8")[:1000])

## Create session-scoped Google Drive folders and archive IDs

Both Fintech and StratLake get separate Drive session folders.

Archive IDs are derived from the active session IDs so restore paths remain collision-resistant and notebook-portable.
M9.1 source-hygiene boundary: Drive folders are session persistence/archive locations only. Keep active Fintech and StratLake runtime work under `/content`, and set `DRIVE_FOLDER_NAME` explicitly before any Drive folder mutation.

In [ ]:
FINTECH_DRIVE_SESSIONS_ROOT = FINTECH_DRIVE_ROOT / "sessions"
STRATLAKE_DRIVE_SESSIONS_ROOT = STRATLAKE_DRIVE_ROOT / "sessions"

FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / FINTECH_SESSION_ID
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / STRATLAKE_SESSION_ID

FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )

FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

for path in [
    FINTECH_DRIVE_SESSION_ROOT,
    STRATLAKE_DRIVE_SESSION_ROOT,
    FINTECH_DRIVE_BACKUP_ROOT,
    STRATLAKE_DRIVE_ARCHIVE_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

FINTECH_ROOT_STR = FINTECH_ROOT.as_posix()
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.as_posix()
MARKETLAKE_ROOT_STR = MARKETLAKE_ROOT.as_posix()
DRIVE_ROOT_STR = DRIVE_ROOT.as_posix()
FINTECH_DRIVE_BACKUP_ROOT_STR = FINTECH_DRIVE_BACKUP_ROOT.as_posix()
STRATLAKE_DRIVE_SESSION_ROOT_STR = STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("FINTECH_ARCHIVE_ID:", FINTECH_ARCHIVE_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)
print("FINTECH_DRIVE_SESSION_ROOT:", FINTECH_DRIVE_SESSION_ROOT)
print("STRATLAKE_DRIVE_SESSION_ROOT:", STRATLAKE_DRIVE_SESSION_ROOT)
print("FINTECH_DRIVE_BACKUP_ROOT:", FINTECH_DRIVE_BACKUP_ROOT)
print("STRATLAKE_DRIVE_ARCHIVE_ROOT:", STRATLAKE_DRIVE_ARCHIVE_ROOT)
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR)
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR)

## Configure Alpaca API credentials from Colab Secrets

This keeps Notebook 06 independently runnable when local curated data is missing and a daily-bars pull is needed.

Recommended Colab Secrets:

```text
ALPACA_API_KEY_ID
ALPACA_API_SECRET_KEY
```

The API key and secret are not printed.

In [ ]:
import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set.")

## Prepare ticker files and Q1 validation window

Notebook 06 uses a compact Q1 demonstration universe.

You can edit `TICKERS` before running ingestion or validation.

In [ ]:
TICKERS = ["AAPL", "MSFT", "NVDA"]
START_DATE = "2025-01-01"
END_DATE = "2025-04-01"

FINTECH_CONFIGS_ROOT = FINTECH_ROOT / "configs"
STRATLAKE_CONFIGS_ROOT = STRATLAKE_ROOT / "configs"
FINTECH_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)
STRATLAKE_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_TICKERS_FILE = FINTECH_CONFIGS_ROOT / "tickers_q1_demo.txt"
STRATLAKE_TICKERS_FILE = STRATLAKE_CONFIGS_ROOT / "tickers_q1_demo.txt"

FINTECH_TICKERS_FILE.write_text("\n".join(TICKERS) + "\n", encoding="utf-8")
STRATLAKE_TICKERS_FILE.write_text("\n".join(TICKERS) + "\n", encoding="utf-8")

DAILY_BARS_ROOT = MARKETLAKE_ROOT / "bars_daily"
DAILY_BARS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_TICKERS_FILE_STR = FINTECH_TICKERS_FILE.as_posix()
STRATLAKE_TICKERS_FILE_STR = STRATLAKE_TICKERS_FILE.as_posix()
DAILY_BARS_ROOT_STR = DAILY_BARS_ROOT.as_posix()

print("TICKERS:", TICKERS)
print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)
print("FINTECH_TICKERS_FILE:", FINTECH_TICKERS_FILE)
print("STRATLAKE_TICKERS_FILE:", STRATLAKE_TICKERS_FILE)
print("DAILY_BARS_ROOT:", DAILY_BARS_ROOT)

## Optional: restore Fintech curated data from Drive before API ingestion

Use this when you already have a previous Fintech archive pack and want to avoid pulling daily bars again.

By default this cell only prints the restore command.
M9.3 note: this preview now uses registry-current Fintech backup-pack restore syntax (`--backup-pack-dir`, `--restore-root`, `--overwrite-policy`). It remains preview/manual guidance only. Static validation covers the command shape but does not execute restore.

In [ ]:
RESTORE_FINTECH_SESSION_ID = FINTECH_SESSION_ID
RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
RESTORE_FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / RESTORE_FINTECH_SESSION_ID / "backups"
RESTORE_FINTECH_BACKUP_PACK_DIR = RESTORE_FINTECH_DRIVE_BACKUP_ROOT / RESTORE_FINTECH_ARCHIVE_ID
RESTORE_FINTECH_BACKUP_PACK_DIR_STR = RESTORE_FINTECH_BACKUP_PACK_DIR.as_posix()

print("Current MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("Current MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())
print("Restore Fintech backup pack:", RESTORE_FINTECH_BACKUP_PACK_DIR)
print("Restore pack exists:", RESTORE_FINTECH_BACKUP_PACK_DIR.exists())

FINTECH_RESTORE_COMMAND_TEXT = (
    "fintech-backup-data restore "
    f"--backup-pack-dir {RESTORE_FINTECH_BACKUP_PACK_DIR_STR} "
    f"--restore-root {MARKETLAKE_ROOT_STR} "
    "--overwrite-policy fail"
)

print("\nFintech restore preview:")
print(FINTECH_RESTORE_COMMAND_TEXT)


## Ensure Q1 daily bars are available locally

This cell makes Notebook 06 independently runnable.

It checks the local curated daily-bars root first. If no Parquet files are found, it runs the Fintech daily-bars backfill using the Alpaca credentials configured above.

In [ ]:
daily_bar_files_before = sorted(DAILY_BARS_ROOT.rglob("*.parquet"))
print("Daily bars before ingestion:", len(daily_bar_files_before))

RUN_DAILY_BACKFILL_IF_MISSING = True

if RUN_DAILY_BACKFILL_IF_MISSING and not daily_bar_files_before:
    !fintech-backfill-daily \
      --symbols {FINTECH_TICKERS_FILE_STR} \
      --start {START_DATE} \
      --end {END_DATE} \
      --out {DAILY_BARS_ROOT_STR} \
      --feed iex \
      --source session_{FINTECH_SESSION_ID} \
      --window month
else:
    print("Skipping daily-bars backfill because files already exist or RUN_DAILY_BACKFILL_IF_MISSING=False.")

## Validate Fintech daily-bars handoff

This validates the local `MARKETLAKE_ROOT` handoff that StratLake will consume.

In [ ]:
import pandas as pd

MARKETLAKE_PARQUET_FILES = sorted(MARKETLAKE_ROOT.rglob("*.parquet"))
DAILY_BAR_FILES = sorted(DAILY_BARS_ROOT.rglob("*.parquet"))

print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())
print("Total MARKETLAKE parquet files:", len(MARKETLAKE_PARQUET_FILES))
print("Daily bars parquet files:", len(DAILY_BAR_FILES))

for path in DAILY_BAR_FILES[:20]:
    print(path)

if not DAILY_BAR_FILES:
    raise FileNotFoundError("No daily bars found. Restore an archive or run the daily-bars backfill cell.")

sample_daily = pd.read_parquet(DAILY_BAR_FILES[0])
print("\nSample file:", DAILY_BAR_FILES[0])
print("Sample shape:", sample_daily.shape)
print("Sample columns:", list(sample_daily.columns))
display(sample_daily.head())

## Optional: archive the Fintech curated Q1 input

This packages the upstream curated daily-bar input into a `{FINTECH_SESSION_ID}`-scoped Drive archive pack.

Default behavior is preview-only. Set `CREATE_FINTECH_ARCHIVE = True` to execute the pack command.
M9.3 note: this preview now uses registry-current Fintech backup-pack syntax (`--workspace-root`, `--source-dataset-root`, `--backup-root`, `--backup-id`, `--shard-size-mb`). It remains preview/manual guidance only. Static validation covers the command shape but does not execute archive creation.

In [ ]:
FINTECH_PACK_COMMAND_TEXT = (
    "fintech-backup-data pack "
    f"--workspace-root {FINTECH_ROOT_STR} "
    f"--source-dataset-root {MARKETLAKE_ROOT_STR} "
    f"--backup-root {FINTECH_DRIVE_BACKUP_ROOT_STR} "
    f"--backup-id {FINTECH_ARCHIVE_ID} "
    "--shard-size-mb 512"
)

print("Fintech archive pack preview:")
print(FINTECH_PACK_COMMAND_TEXT)

CREATE_FINTECH_ARCHIVE = False

if CREATE_FINTECH_ARCHIVE:
    subprocess.run(FINTECH_PACK_COMMAND_TEXT.split(), check=True)
else:
    print("Preview only. Set CREATE_FINTECH_ARCHIVE=True to create the archive pack.")


## Ensure StratLake feature outputs are available

Notebook 06 should mainly validate, not repeatedly rebuild. But this cell makes the notebook independently runnable by building features if no local feature Parquet files are found.

In [ ]:
feature_candidates_before = sorted((STRATLAKE_ROOT / "data").rglob("*.parquet"))
print("Feature parquet files before build:", len(feature_candidates_before))

RUN_FEATURE_BUILD_IF_MISSING = True

if RUN_FEATURE_BUILD_IF_MISSING and not feature_candidates_before:
    os.chdir(STRATLAKE_ROOT)
    print("Current working directory:", Path.cwd())
    print("Using MARKETLAKE_ROOT:", MARKETLAKE_ROOT_STR)

    !stratlake-build-features \
      --timeframe 1D \
      --start {START_DATE} \
      --end {END_DATE} \
      --tickers {STRATLAKE_TICKERS_FILE_STR} \
      --marketlake-root {MARKETLAKE_ROOT_STR}
else:
    print("Skipping feature build because feature files already exist or RUN_FEATURE_BUILD_IF_MISSING=False.")

## Validate StratLake feature outputs

This cell inspects generated feature files and reads a sample Parquet file to catch obvious handoff/schema problems.

In [ ]:
feature_candidates = sorted((STRATLAKE_ROOT / "data").rglob("*.parquet"))
artifact_candidates = sorted((STRATLAKE_ROOT / "artifacts").rglob("*")) if (STRATLAKE_ROOT / "artifacts").exists() else []

print("Generated/available StratLake parquet files:", len(feature_candidates))
for path in feature_candidates[:30]:
    print(path)

if not feature_candidates:
    raise FileNotFoundError("No StratLake feature Parquet files found. Run the feature build cell or restore a StratLake archive.")

sample_feature = pd.read_parquet(feature_candidates[0])
print("\nSample feature file:", feature_candidates[0])
print("Sample shape:", sample_feature.shape)
print("Sample columns:", list(sample_feature.columns))
display(sample_feature.head())

print("\nArtifact file/directory count:", len(artifact_candidates))
for path in artifact_candidates[:30]:
    print(path)

## Validate session portability assumptions

This checks the key Notebook 04/05/06 portability assumptions:

```text
1. Fintech and StratLake session IDs are separate
2. MARKETLAKE_ROOT exists and is explicit
3. universe.yml and paths.yml are present
4. Drive archive roots are session-scoped
```

In [ ]:
checks = {
    "fintech_session_id_present": bool(FINTECH_SESSION_ID),
    "stratlake_session_id_present": bool(STRATLAKE_SESSION_ID),
    "session_ids_are_distinct": FINTECH_SESSION_ID != STRATLAKE_SESSION_ID,
    "marketlake_root_exists": MARKETLAKE_ROOT.exists(),
    "universe_yml_exists": (STRATLAKE_ROOT / "configs" / "universe.yml").exists(),
    "paths_yml_exists": (STRATLAKE_ROOT / "configs" / "paths.yml").exists(),
    "fintech_drive_backup_root_exists": FINTECH_DRIVE_BACKUP_ROOT.exists(),
    "stratlake_drive_archive_root_exists": STRATLAKE_DRIVE_ARCHIVE_ROOT.exists(),
    "daily_bars_present": bool(DAILY_BAR_FILES),
    "feature_files_present": bool(feature_candidates),
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise AssertionError("Failed portability/session checks: " + ", ".join(failed))

## Preview StratLake session export

This dry run previews a lightweight Drive session export that includes features, artifacts, and configs.

In [ ]:
!stratlake-session-export \
  --root {STRATLAKE_ROOT_STR} \
  --drive-root {STRATLAKE_DRIVE_SESSION_ROOT_STR} \
  --include-features \
  --include-artifacts \
  --include-configs \
  --dry-run

## Optional: archive the StratLake feature session

This is the stronger checkpoint after validation.

It should include:

```text
features
artifacts
configs, including universe.yml and paths.yml
```

Default behavior is preview-only. Set `CREATE_STRATLAKE_ARCHIVE = True` to execute the archive bootstrap command.
M9.3 note: StratLake archive/bootstrap preview commands remain optional/unverified manual guidance. Upstream contract has not been verified. Do not treat this source as validation of those command contracts.

In [ ]:
stratlake_archive_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT_STR,
    "--archive-id", STRATLAKE_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT_STR,
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("StratLake archive bootstrap command:")
print(" \\\n  ".join(stratlake_archive_cmd))

CREATE_STRATLAKE_ARCHIVE = False

if CREATE_STRATLAKE_ARCHIVE:
    subprocess.run(stratlake_archive_cmd, check=True)
else:
    print("Preview only. Set CREATE_STRATLAKE_ARCHIVE=True to create the StratLake archive pack.")

## Restore-readiness command preview

Do not restore over the active workspace by default.

This cell prints the restore command for the active `{STRATLAKE_SESSION_ID}` archive. For a previous session, update `RESTORE_STRATLAKE_SESSION_ID` and `RESTORE_STRATLAKE_ARCHIVE_ID` first.
M9.3 note: StratLake archive/bootstrap preview commands remain optional/unverified manual guidance. Upstream contract has not been verified. Do not treat this source as validation of those command contracts.

In [ ]:
RESTORE_STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID
RESTORE_STRATLAKE_ARCHIVE_ID = f"stratlake-session-{RESTORE_STRATLAKE_SESSION_ID}"
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / RESTORE_STRATLAKE_SESSION_ID / "archives"
RESTORE_STRATLAKE_ARCHIVE_PACK_DIR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT / RESTORE_STRATLAKE_ARCHIVE_ID
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()

stratlake_restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--root", STRATLAKE_ROOT_STR,
    "--archive-id", RESTORE_STRATLAKE_ARCHIVE_ID,
    "--drive-root", RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT_STR,
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("Restore archive pack:", RESTORE_STRATLAKE_ARCHIVE_PACK_DIR)
print("Restore pack exists:", RESTORE_STRATLAKE_ARCHIVE_PACK_DIR.exists())
print("\nStratLake archive restore command:")
print(" \\\n  ".join(stratlake_restore_cmd))

## Final handoff summary

Notebook 06 validates that the latest Fintech â†’ StratLake notebook workflow produced usable local feature data and session-scoped Drive checkpoint paths.

In [ ]:
summary = {
    "fintech": {
        "session_id": FINTECH_SESSION_ID,
        "marketlake_root": MARKETLAKE_ROOT.as_posix(),
        "daily_bar_file_count": len(DAILY_BAR_FILES),
        "drive_session_root": FINTECH_DRIVE_SESSION_ROOT.as_posix(),
        "archive_id": FINTECH_ARCHIVE_ID,
        "archive_pack_dir": FINTECH_BACKUP_PACK_DIR.as_posix(),
    },
    "stratlake": {
        "session_id": STRATLAKE_SESSION_ID,
        "feature_file_count": len(feature_candidates),
        "drive_session_root": STRATLAKE_DRIVE_SESSION_ROOT.as_posix(),
        "archive_id": STRATLAKE_ARCHIVE_ID,
        "archive_pack_dir": STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
        "universe_yml": (STRATLAKE_ROOT / "configs" / "universe.yml").as_posix(),
        "paths_yml": (STRATLAKE_ROOT / "configs" / "paths.yml").as_posix(),
    },
    "next_notebook": "Notebook 07 â€” Strategy Backtest / Feature Consumption / Research Workflow",
}

print(json.dumps(summary, indent=2))